In [1]:
from sympy.ntheory import factorint
from tools import *

import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np
import pickle
import torch as tn
import torchtt as tt

In [2]:
test_case = 'ltc1'
mesh_num = 5
filename = f'io/out_{test_case}_cvt_{mesh_num}.nc'
data = nc.Dataset(filename)
print(data)

ncells = data.dimensions['nCells'].size
nedges = data.dimensions['nEdges'].size
print(f'ncells = {ncells}')
print(f'nedges = {nedges}')

ntimeLevels = data.dimensions['Time'].size
print(f'ntimeLevels = {ntimeLevels}')

hh_cell = data.variables['hh_cell'][:, :, 0]
uu_edge = data.variables['uu_edge'][:, :, 0]

data.close()

<class 'netCDF4._netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    on_a_sphere: YES
    sphere_radius: 6371220.0
    is_periodic: NO
    source: swe-python
    dimensions(sizes): Time(15), Step(15), TWO(2), nCells(10242), nEdges(30720), nVertices(20480), nVertLevels(1), maxEdges(6), maxEdges2(12), vertexDegree(3)
    variables(dimensions): float64 lonCell(nCells), float64 latCell(nCells), float64 xCell(nCells), float64 yCell(nCells), float64 zCell(nCells), float64 areaCell(nCells), int32 verticesOnCell(nCells, maxEdges), int32 edgesOnCell(nCells, maxEdges), int32 cellsOnCell(nCells, maxEdges), int32 nEdgesOnCell(nCells), float64 lonEdge(nEdges), float64 latEdge(nEdges), float64 xEdge(nEdges), float64 yEdge(nEdges), float64 zEdge(nEdges), float64 dvEdge(nEdges), float64 dcEdge(nEdges), int32 verticesOnEdge(nEdges, TWO), float64 weightsOnEdge(nEdges, maxEdges2), int32 cellsOnEdge(nEdges, TWO), int32 edgesOnEdge(nEdges, maxEdges2), int32 nEdgesOnEdge(nEdges), float6

In [3]:
prime_factors_cell = list( factorint(ncells).items() )
prime_factors_edge = list( factorint(nedges).items() )
print(f'prime_factors_cell = {prime_factors_cell}')
print(f'prime_factors_edge = {prime_factors_edge}')

tt_tens_shape_cell = []
for factor in prime_factors_cell:
    #tt_tens_shape_cell += [ factor[0] ] * factor[1]
    tt_tens_shape_cell += [ factor[0] ** factor[1] ]
# END for

tt_tens_shape_edge = []
for factor in prime_factors_edge:
    #tt_tens_shape_edge += [ factor[0] ] * factor[1]
    tt_tens_shape_edge += [ factor[0] ** factor[1] ]
# END for

print(f'tt_tens_shape_cell = {tt_tens_shape_cell}')
print(f'tt_tens_shape_edge = {tt_tens_shape_edge}')

prime_factors_cell = [(2, 1), (3, 2), (569, 1)]
prime_factors_edge = [(2, 11), (3, 1), (5, 1)]
tt_tens_shape_cell = [2, 9, 569]
tt_tens_shape_edge = [2048, 3, 5]


In [4]:
eps_h = 1e-8
eps_u = 1e-8

h = np.zeros([ntimeLevels, ncells, 1])
u = np.zeros([ntimeLevels, nedges, 1])

for level in range(ntimeLevels):
    h_tt = tt.TT(hh_cell[level, :],
                 tt_tens_shape_cell,
                 eps=eps_h)
    u_tt = tt.TT(uu_edge[level, :],
                 tt_tens_shape_edge,
                 eps=eps_u)

    h[level, :, 0] = h_tt.full().numpy().flatten()
    u[level, :, 0] = u_tt.full().numpy().flatten()
# END for

In [5]:
h_name = f'h_tt_{eps_h}'
u_name = f'u_tt_{eps_u}'

data = nc.Dataset(filename, 'a', format='NETCDF4')

try:
    data.createVariable(h_name, 'f8', ('Time', 'nCells', 'nVertLevels'))
    print(f'created variable {h_name}')
except:
    print(f'variable {h_name} already exists')
# END try

try:
    data.createVariable(u_name, 'f8', ('Time', 'nEdges', 'nVertLevels'))
    print(f'created variable {u_name}')
except:
    print(f'variable {u_name} already exists')
# END try

data.variables[h_name][:, :, 0] = h
data.variables[u_name][:, :, 0] = u

data.close()

variable h_tt_1e-08 already exists
variable u_tt_1e-08 already exists


In [6]:
data = nc.Dataset(filename, 'a', format='NETCDF4')

cells_on_cell = data.variables['cellsOnCell']
visited = np.zeros(ncells, dtype=int)
root_ind = 0

sorted_cell_inds = []
queue = [root_ind]
visited[root_ind] = 1
while queue:
    cur_ind = queue.pop(0)
    sorted_cell_inds.append(cur_ind)

    adjacent_inds = np.array(cells_on_cell[cur_ind][cells_on_cell[cur_ind] != 0]) - 1
    for ind in adjacent_inds:
        if not visited[ind]:
            queue.append(ind)
            visited[ind] = 1
        # END if
    # END for
# END while   
sorted_cell_inds = np.array(sorted_cell_inds)
print(sorted_cell_inds)

bfs_order_name_cell = 'bfs_order_cell'
try:
    data.createVariable(bfs_order_name_cell, 'u8', ('nCells'))
    print(f'created variable {bfs_order_name_cell}')
except:
    print(f'variable {bfs_order_name_cell} already exists')
# END try
data.variables[bfs_order_name_cell][sorted_cell_inds] = np.arange(ncells)

data.close()

[   0 2563 2562 ... 2567 2571    1]
variable bfs_order_cell already exists


In [7]:
data = nc.Dataset(filename, 'a', format='NETCDF4')

edges_on_edge = data.variables['edgesOnEdge']
visited = np.zeros(nedges, dtype=int)
root_ind = 0

sorted_edge_inds = []
queue = [root_ind]
visited[root_ind] = 1
while queue:
    cur_ind = queue.pop(0)
    sorted_edge_inds.append(cur_ind)

    adjacent_inds = np.array(edges_on_edge[cur_ind][edges_on_edge[cur_ind] != 0]) - 1
    for ind in adjacent_inds:
        if not visited[ind]:
            queue.append(ind)
            visited[ind] = 1
        # END if
    # END for
# END while   
sorted_edge_inds = np.array(sorted_edge_inds)
print(sorted_edge_inds)

bfs_order_name_edge = 'bfs_order_edge'
try:
    data.createVariable(bfs_order_name_edge, 'u8', ('nEdges'))
    print(f'created variable {bfs_order_name_edge}')
except:
    print(f'variable {bfs_order_name_edge} already exists')
# END try
data.variables[bfs_order_name_edge][sorted_edge_inds] = np.arange(nedges)

data.close()

[    0 22412 22416 ...    10 24024  5383]
variable bfs_order_edge already exists


In [8]:
def order_by_angle(ds, root, adjacent):
    root_pt = np.array([ds.variables['xCell'][root],
                        ds.variables['yCell'][root]])
    root_vec = np.array([1, 0])

    angles = np.zeros(adjacent.size)
    for i, ind in enumerate(adjacent):
        pt = np.array([ds.variables['xCell'][ind],
                       ds.variables['yCell'][ind]])
        vec = pt - root_pt
        
        angle = np.arccos(np.dot(root_vec, vec) / np.linalg.norm(vec))
        if vec[1] < 0:
            angle = 2 * np.pi - angle
        # END if
        angles[i] = angle
    # END for
    
    sorted_inds = np.argsort(angles)
    return adjacent[sorted_inds], angles[sorted_inds]
# END order_by_angle

In [14]:
ds = nc.Dataset(filename, 'a', format='NETCDF4')
cells_on_cell = ds.variables['cellsOnCell']

visited = np.zeros(ncells, dtype=int)
halo = [[], []]

start_ind = 0
visited[start_ind] = 1
sorted_inds = [start_ind]
halo[0].append(start_ind)

iteration = 0
while halo[0] or halo[1]:
    cur_halo = iteration % 2
    other_halo = (iteration + 1) % 2

    # sweep top half of cells adjacent to the cell
    # at the end of the current halo
    root = halo[cur_halo][-1]
    adjacent = np.array(cells_on_cell[root][cells_on_cell[root] != 0]) - 1
    adjacent, angles = order_by_angle(ds, root, adjacent)
    for ind, angle in zip(adjacent, angles):
        if angle < np.pi and not visited[ind]:
            #print(f'1root = {root}, ind = {ind}')
            visited[ind] = 1
            sorted_inds.append(ind)
            halo[other_halo].append(ind)
        # END if
    # END for

    # sweep all cells adjacent to the cells in 
    # interior of the current halo
    for root in halo[cur_halo][:-1]:
        adjacent = np.array(cells_on_cell[root][cells_on_cell[root] != 0]) - 1
        adjacent, angles = order_by_angle(ds, root, adjacent)
        #print(f'root = {root}\n{adjacent}\n{angles}')
        for ind, angle in zip(adjacent, angles):
            if not visited[ind]:
                #print(f'2root = {root}, ind = {ind}')
                visited[ind] = 1
                sorted_inds.append(ind)
                halo[other_halo].append(ind)
            # END if
        # END for
    # END for
    
    # sweep bottom half of cells adjacent to the cell
    # at the end of the current halo
    root = halo[cur_halo][-1]
    adjacent = np.array(cells_on_cell[root][cells_on_cell[root] != 0]) - 1
    adjacent, angles = order_by_angle(ds, root, adjacent)
    for ind, angle in zip(adjacent, angles):
        if angle >= np.pi and not visited[ind]:
            #print(f'3root = {root}, ind = {ind}')
            visited[ind] = 1
            sorted_inds.append(ind)
            halo[other_halo].append(ind)
        # END if
    # END for

    #print(f'iteration = {iteration}')
    #print(sorted_inds)
    #print(halo[cur_halo])
    #print(halo[other_halo])
    #print('')
    halo[cur_halo][:] = []
    iteration += 1

    #if iteration == 3: break
# END while
sorted_cell_inds = np.array(sorted_inds)
print(sorted_cell_inds)

[   0 2562 2563 ... 2567 2568    1]


In [15]:
spiral_order_name_cell = 'spiral_order_cell'
try:
    ds.createVariable(spiral_order_name_cell, 'u8', ('nCells'))
    print(f'created variable {spiral_order_name_cell}')
except:
    print(f'variable {spiral_order_name_cell} already exists')
# END try
ds.variables[spiral_order_name_cell][sorted_cell_inds] = np.arange(ncells)

ds.close()

variable spiral_order_cell already exists
